In [ ]:
from pathlib import Path

import logfire

from src.common.cache.embedding_cache import EmbeddingCache
from src.common.services.qdrant import QdrantStorageService
from src.common.storage.storage_factory import StorageFactory
from src.common.utils.config import config
from src.common.utils.constants import ParseMethod, StorageType
from src.common.utils.helper import separate_content
from src.common.utils.tokenizer import TikTokenTokenizer
from src.ingestion.chunking.chunk import build_parent_child_chunk
from src.ingestion.chunking.chunker_factory import create_chunker
from src.ingestion.chunking.chunking_config import ChunkingConfig
from src.ingestion.embedding import EmbeddingService
from src.ingestion.processor import Processor

In [ ]:
logfire.configure(service_name="parsing")

In [ ]:
cwd = Path.cwd().parent
file_path = cwd / "data/Attention-is_all_you_need.pdf"

In [ ]:
local_config = {
    "type": StorageType.LOCAL.value,
    "base_dir": config.STORAGE_BASE_DIR,
}

in_storage = StorageFactory.create(local_config)

In [ ]:
tokenizer = TikTokenTokenizer()
emb_cache = await EmbeddingCache.create(dsn=config.POSTGRES_CONN_STRING, max_entries=50_000)
embedding_service = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME,
    dimensions=config.EMBEDDING_DIMENSIONS,
    batch_size=config.EMBEDDING_BATCH_SIZE,
    cache=emb_cache,
)

storage_service = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=embedding_service.vector_size,
    collection_name=config.QDRANT_COLLECTION_NAME,
)
processor = Processor(tokenizer, embedding_service, storage_service)

In [ ]:
out, doc_id = await processor.process_document(
    file_path=str(file_path), parse_method=ParseMethod.DOCLING
)

In [ ]:
print(out)

In [ ]:
text_content, multimodal_items = separate_content(out)

In [ ]:
multimodal_items

In [ ]:
chunking_config = ChunkingConfig(type="recursive_character", size=512, overlap=64)
chunker = create_chunker(chunking_config)
text_chunks = chunker.chunk(text_content)
multimodel_chunks = chunker.chunk_multimodal_items(
    multimodal_items, doc_id=doc_id, source_file=str(file_path), start_index=len(text_chunks)
)

In [ ]:
chunks = text_chunks + multimodel_chunks

In [ ]:
len(chunks)

In [ ]:
enriched_chunks = build_parent_child_chunk(chunks, tokenizer)

In [ ]:
enriched_chunks

In [ ]:
in_storage.upload(key="chunks", data=chunks)

In [ ]:
emb_chunks = await embedding_service.embed_chunks(enriched_chunks)

In [ ]:
in_storage.upload(key="embedded_chunks", data=emb_chunks)

In [ ]:
result = await processor.ingest_document(file_path=file_path, parse_method=ParseMethod.DOCLING)

In [ ]:
result